# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset on protein analysis using the `mlcroissant` library, following the [Croissant](https://mlcommons.org/croissant/) schema.

### Dataset Source
The dataset is accessed via its Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata: name, description, published date
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review the available record sets in the dataset and their field/column structure.

We reference each entity by its `@id` field according to the Croissant schema best practice.

In [ ]:
# List all record sets by their @id and name
all_record_sets = list(dataset.record_sets)
print(f"Found {len(all_record_sets)} record sets:")
record_set_ids = []
for rs in all_record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    record_set_ids.append(rs['@id'])

In [ ]:
# For each record set, show fields (columns) and their @id
for rs in all_record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        if isinstance(f, dict):
            f_id = f.get('@id', 'N/A')
            f_name = f.get('name', 'N/A')
        else:  # f is a string (just @id)
            f_id = f
            f_name = 'N/A'
        print(f"   Field @id: {f_id} | name: {f_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s as extracted above.

In [ ]:
# For demonstration, use the first record set
chosen_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"Using record set: {chosen_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from {record_set_id}")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

# Overview DataFrame columns
if chosen_record_set_id is not None and chosen_record_set_id in dataframes:
    print(f"Fields/columns for record set {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate basic EDA: filtering, normalization, and grouping. Field references are always by their `@id`.

In [ ]:
# Choose a numeric field, e.g., 'cr:MW_kDa' (molecular weight in kiloDaltons), if present
possible_numeric_fields = [c for c in dataframes[chosen_record_set_id].columns if 'MW' in c or 'abundance' in c or 'count' in c or 'percentage' in c]
print(f"Possible numeric fields: {possible_numeric_fields}")
numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else dataframes[chosen_record_set_id].columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Basic cleaning: Try to convert to numeric
df = dataframes[chosen_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} found\n")
display(filtered_df.head(3))

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
print(f"Normalized {numeric_field_id} (z-score normalization):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Group by a categorical field if available (e.g., 'cr:accession', 'cr:description')
possible_group_fields = [c for c in df.columns if 'accession' in c or 'description' in c or 'sample' in c or 'group' in c]
group_field_id = possible_group_fields[0] if possible_group_fields else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by group field: {group_field_id}")
    display(grouped_df.head(3))

## 5. Visualization

Visualize data distributions or relationships (numeric field distribution, grouped means, scatterplots, etc).

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(7,4))
df[numeric_field_id].hist(bins=30)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot for group/category if available
if group_field_id:
    top_groups = df[group_field_id].value_counts().head(10).index.tolist()
    subset = df[df[group_field_id].isin(top_groups)]
    plt.figure(figsize=(10,5))
    subset.boxplot(column=numeric_field_id, by=group_field_id, rot=45)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We successfully loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library referencing all entities by their `@id`.
- We explored the dataset structure, identified record sets and available fields, and extracted the main data table.
- Essential exploratory data analysis steps (filtering, normalization, grouping) and visualizations (histogram, boxplot) were demonstrated on key numeric fields such as molecular weight or abundance.

You are encouraged to consult the Croissant schema and [mlcroissant documentation](https://pypi.org/project/mlcroissant/) for more advanced analyses.